# PadChest Database analysis

In [1]:
import os
import shutil
import pandas as pd

## 1. Save all the images in the same folder

### 1.1 Obtain the image paths

In [24]:
# Main folder that contains the images
root_folder = '../../TrainXray/Data/BIMCV-PadChest-FULL-resized512'

# List to store the paths of the images
images = []

# We walk the structure of folders and subfolders with os.walk()
for root, dirs, files in os.walk(root_folder):
    for file in files:
        # We check if the file is an image
        if file.endswith(".png"):
            # If it is an image, we add the path to the list
            image_path = os.path.join(root, file)
            images.append(image_path)
            
# Print the length of the list of image paths
print("Number of images:", len(images))

Number of images: 157105


### 1.1 Create a new folder with all the images

In [17]:
# Create the destination folder if it does not already exist
dest_path = '../../../TrainXray/Data/PadChest512'
if not os.path.exists(dest_path):
    os.makedirs(dest_path)

# Copy each image to the destination folder
for image in images:
    shutil.copy(image, dest_path)

## 2. Analyze the number of images: csv file

In [3]:
csv_table = pd.read_csv('../../../TrainXray/Data/PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv', low_memory = False)

### 2.1 MethodLabel = "RNN_model": Different projections

In [4]:
# MethodLabel = RNN_model
csv_table_RNN_model = csv_table[csv_table['MethodLabel'] == 'RNN_model']
print('MethodLabel="RNN_model":', len(csv_table_RNN_model))

# Loop over projections
projections = ['PA', 'AP', 'AP_horizontal', 'L', 'COSTAL', 'UNK', 'EXCLUDE']
for projection in projections:
    csv_table_filtered = csv_table_RNN_model[csv_table_RNN_model['Projection'] == projection]
    print(f'MethodLabel="RNN_model" and Projection="{projection}":', len(csv_table_filtered))

MethodLabel="RNN_model": 121808
MethodLabel="RNN_model" and Projection="PA": 67767
MethodLabel="RNN_model" and Projection="AP": 3984
MethodLabel="RNN_model" and Projection="AP_horizontal": 12450
MethodLabel="RNN_model" and Projection="L": 36971
MethodLabel="RNN_model" and Projection="COSTAL": 617
MethodLabel="RNN_model" and Projection="UNK": 8
MethodLabel="RNN_model" and Projection="EXCLUDE": 11


### 2.2 MethodLabel = "Physician": Different projections

In [5]:
# MethodLabel = Physician 
csv_table_physician = csv_table[csv_table['MethodLabel'] == 'Physician']
print('MethodLabel="Physician":', len(csv_table_physician))

# Loop over projections
projections = ['PA', 'AP', 'AP_horizontal', 'L', 'COSTAL', 'UNK', 'EXCLUDE']
for projection in projections:
    csv_table_filtered = csv_table_physician[csv_table_physician['Projection'] == projection]
    print(f'MethodLabel="Physician" and Projection="{projection}":', len(csv_table_filtered))

MethodLabel="Physician": 39053
MethodLabel="Physician" and Projection="PA": 23961
MethodLabel="Physician" and Projection="AP": 575
MethodLabel="Physician" and Projection="AP_horizontal": 1896
MethodLabel="Physician" and Projection="L": 12608
MethodLabel="Physician" and Projection="COSTAL": 13
MethodLabel="Physician" and Projection="UNK": 0
MethodLabel="Physician" and Projection="EXCLUDE": 0


### 2.3 MethodLabel = "Physician" and Projection = "PA". Normal and Abnormal labels

In [6]:
csv_filtered = csv_table[(csv_table['MethodLabel'] == 'Physician') & (csv_table['Projection'] == 'PA')]
print('Method Label="Physician" and Projection="PA":', len(csv_filtered))

Method Label="Physician" and Projection="PA": 23961


In [7]:
# Filter the dataset to only include rows where the 'Labels' column has a certain number of occurrences of the string 'normal'
normal_labels = [row for row in csv_filtered['Labels'] if row.count("'normal'") == len(eval(row))]
csv_filtered_normal = csv_filtered[csv_filtered['Labels'].isin(normal_labels)]
print('Method Label="Physician", Projection="PA" and Labels="normal":', len(csv_filtered_normal))

Method Label="Physician", Projection="PA" and Labels="normal": 8708


In [8]:
# Check results
for row in csv_filtered['Labels'].head(20).iloc:
    print(row)
    print("Normal labels:",row.count("'normal'"))
    print("Total Labels:",len(eval(row)))
    print("----------------")

['normal']
Normal labels: 1
Total Labels: 1
----------------
['pulmonary fibrosis', 'chronic changes', 'kyphosis', 'pseudonodule', 'ground glass pattern']
Normal labels: 0
Total Labels: 5
----------------
['chronic changes']
Normal labels: 0
Total Labels: 1
----------------
['laminar atelectasis', 'pleural effusion', 'apical pleural thickening', 'suture material', 'sternotomy']
Normal labels: 0
Total Labels: 5
----------------
['superior mediastinal enlargement', 'aortic elongation', 'scoliosis', 'sclerotic bone lesion', 'supra aortic elongation', 'vertebral degenerative changes', 'goiter']
Normal labels: 0
Total Labels: 7
----------------
['normal']
Normal labels: 1
Total Labels: 1
----------------
['normal']
Normal labels: 1
Total Labels: 1
----------------
['COPD signs', 'air trapping']
Normal labels: 0
Total Labels: 2
----------------
['descendent aortic elongation', 'scoliosis', 'aortic atheromatosis']
Normal labels: 0
Total Labels: 3
----------------
['normal']
Normal labels: 1
T

In [32]:
# Filter the dataset to only include rows where the 'Labels' column include a label different from 'normal'
abnormal_labels = [row for row in csv_filtered['Labels'] if row.count("'normal'") != len(eval(row))]
csv_filtered_abnormal = csv_filtered[csv_filtered['Labels'].isin(abnormal_labels)]
print('Method Label="Physician", Projection="PA" and Labels="abnormal":', len(csv_filtered_abnormal))

Method Label="Physician", Projection="PA" and Labels="abnormal": 15253


## -----------------------------------------------------------------------------------------------------------

## 3. Analyze the number of images: PadChest(512x512) 

### 3.1 Obtain the image IDs available in Ebatinca Dataset and store them in a txt file

In [33]:
# Directory where the images are located
images_path = '../../../TrainXray/Data/PadChest512'

# List to store available image IDs
avail_imgID = []

# Iterate over the files in the directory
for file in os.listdir(images_path):
    # Check if the file is an image
    if file.endswith(".png"):
        # Add the name of the image to the list
        avail_imgID.append(file)

# Print the number of images
print("Number of images:",len(avail_imgID))

Number of images: 157105


In [32]:
# Write the avail_imgID list in a txt file
output_file = "available_imageIDs_PadChest512.txt"

# Escribir la lista en el archivo de salida
with open(output_file, "w") as f:
    for imageID in avail_imgID:
        f.write(imageID + "\n")

### 3.2 Filter the labels in the csv file by our available images and repeat the steps in section 2

In [34]:
# Name of the input file with the available image IDs
input_file = "available_imageIDs_PadChest512.txt"

# Read the input file and save its contents to a list
with open(input_file, "r") as f:
    avail_imgID = [line.strip() for line in f.readlines()]

In [35]:
csv_table = pd.read_csv('../../../TrainXray/Data/PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv', low_memory = False)

# Filter by available image IDs
csv_table = csv_table[csv_table["ImageID"].isin(avail_imgID)]

In [36]:
# MethodLabel = RNN_model
csv_table_RNN_model = csv_table[csv_table['MethodLabel'] == 'RNN_model']
print('MethodLabel="RNN_model":', len(csv_table_RNN_model))

# Loop over projections
projections = ['PA', 'AP', 'AP_horizontal', 'L', 'COSTAL', 'UNK', 'EXCLUDE']
for projection in projections:
    csv_table_filtered = csv_table_RNN_model[csv_table_RNN_model['Projection'] == projection]
    print(f'MethodLabel="RNN_model" and Projection="{projection}":', len(csv_table_filtered))

MethodLabel="RNN_model": 121768
MethodLabel="RNN_model" and Projection="PA": 67747
MethodLabel="RNN_model" and Projection="AP": 3982
MethodLabel="RNN_model" and Projection="AP_horizontal": 12448
MethodLabel="RNN_model" and Projection="L": 36963
MethodLabel="RNN_model" and Projection="COSTAL": 617
MethodLabel="RNN_model" and Projection="UNK": 0
MethodLabel="RNN_model" and Projection="EXCLUDE": 11


In [37]:
# MethodLabel = Physician 
csv_table_physician = csv_table[csv_table['MethodLabel'] == 'Physician']
print('MethodLabel="Physician":', len(csv_table_physician))

# Loop over projections
projections = ['PA', 'AP', 'AP_horizontal', 'L', 'COSTAL', 'UNK', 'EXCLUDE']
for projection in projections:
    csv_table_filtered = csv_table_physician[csv_table_physician['Projection'] == projection]
    print(f'MethodLabel="Physician" and Projection="{projection}":', len(csv_table_filtered))

MethodLabel="Physician": 35337
MethodLabel="Physician" and Projection="PA": 21701
MethodLabel="Physician" and Projection="AP": 519
MethodLabel="Physician" and Projection="AP_horizontal": 1699
MethodLabel="Physician" and Projection="L": 11408
MethodLabel="Physician" and Projection="COSTAL": 10
MethodLabel="Physician" and Projection="UNK": 0
MethodLabel="Physician" and Projection="EXCLUDE": 0


In [38]:
csv_filtered = csv_table[(csv_table['MethodLabel'] == 'Physician') & (csv_table['Projection'] == 'PA')]
print('Method Label="Physician" and Projection="PA":', len(csv_filtered))

Method Label="Physician" and Projection="PA": 21701


In [39]:
# Filter the dataset to only include rows where the 'Labels' column has a certain number of occurrences of the string 'normal'
normal_labels = [row for row in csv_filtered['Labels'] if row.count("'normal'") == len(eval(row))]
csv_filtered_normal = csv_filtered[csv_filtered['Labels'].isin(normal_labels)]
print('Method Label="Physician", Projection="PA" and Labels="normal":', len(csv_filtered_normal))

Method Label="Physician", Projection="PA" and Labels="normal": 7871


In [40]:
# Filter the dataset to only include rows where the 'Labels' column include a label different from 'normal'
abnormal_labels = [row for row in csv_filtered['Labels'] if  row.count("'normal'") != len(eval(row))]
csv_filtered_abnormal = csv_filtered[csv_filtered['Labels'].isin(abnormal_labels)]
print('Method Label="Physician", Projection="PA" and Labels="abnormal":', len(csv_filtered_abnormal))

Method Label="Physician", Projection="PA" and Labels="abnormal": 13830


## 4. Analyze the number of images: PadChest(224x24)

### 4.1 Remove missing files in PadChest (224x224)

In [41]:
csv_table = pd.read_csv('../../../TrainXray/Data/PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv', low_memory = False)

# Remove missing files
missing1 = ["216840111366964012819207061112010307142602253_04-014-084.png",
           "216840111366964012989926673512011074122523403_00-163-058.png",
           "216840111366964012959786098432011033083840143_00-176-115.png",
           "216840111366964012558082906712009327122220177_00-102-064.png",
           "216840111366964012339356563862009072111404053_00-043-192.png",
           "216840111366964013076187734852011291090445391_00-196-188.png",
           "216840111366964012373310883942009117084022290_00-064-025.png", 
           "216840111366964012283393834152009033102258826_00-059-087.png",
           "216840111366964012373310883942009170084120009_00-097-074.png",
           "216840111366964012819207061112010315104455352_04-024-184.png",
           "216840111366964012819207061112010306085429121_04-020-102.png"]
        
missing2 = ["216840111366964013076187734852011178154626671_00-145-086.png",
            "216840111366964012339356563862009068084200743_00-045-105.png",
            "216840111366964012989926673512011101154138555_00-191-086.png",
            "216840111366964012989926673512011132200139442_00-157-099.png",
            "216840111366964012558082906712009300162151055_00-078-079.png"]

csv_table = csv_table[~csv_table["ImageID"].isin(missing1)]
csv_table = csv_table[~csv_table["ImageID"].isin(missing2)]

In [42]:
len(csv_table)

160845

### 4.2 Repeat the steps in sections 2 and 3

In [43]:
# MethodLabel = RNN_model
csv_table_RNN_model = csv_table[csv_table['MethodLabel'] == 'RNN_model']
print('MethodLabel="RNN_model":', len(csv_table_RNN_model))

# Loop over projections
projections = ['PA', 'AP', 'AP_horizontal', 'L', 'COSTAL', 'UNK', 'EXCLUDE']
for projection in projections:
    csv_table_filtered = csv_table_RNN_model[csv_table_RNN_model['Projection'] == projection]
    print(f'MethodLabel="RNN_model" and Projection="{projection}":', len(csv_table_filtered))

MethodLabel="RNN_model": 121792
MethodLabel="RNN_model" and Projection="PA": 67762
MethodLabel="RNN_model" and Projection="AP": 3982
MethodLabel="RNN_model" and Projection="AP_horizontal": 12449
MethodLabel="RNN_model" and Projection="L": 36968
MethodLabel="RNN_model" and Projection="COSTAL": 617
MethodLabel="RNN_model" and Projection="UNK": 3
MethodLabel="RNN_model" and Projection="EXCLUDE": 11


In [44]:
# MethodLabel = Physician 
csv_table_physician = csv_table[csv_table['MethodLabel'] == 'Physician']
print('MethodLabel="Physician":', len(csv_table_physician))

# Loop over projections
projections = ['PA', 'AP', 'AP_horizontal', 'L', 'COSTAL', 'UNK', 'EXCLUDE']
for projection in projections:
    csv_table_filtered = csv_table_physician[csv_table_physician['Projection'] == projection]
    print(f'MethodLabel="Physician" and Projection="{projection}":', len(csv_table_filtered))

MethodLabel="Physician": 39053
MethodLabel="Physician" and Projection="PA": 23961
MethodLabel="Physician" and Projection="AP": 575
MethodLabel="Physician" and Projection="AP_horizontal": 1896
MethodLabel="Physician" and Projection="L": 12608
MethodLabel="Physician" and Projection="COSTAL": 13
MethodLabel="Physician" and Projection="UNK": 0
MethodLabel="Physician" and Projection="EXCLUDE": 0


In [45]:
csv_filtered = csv_table[(csv_table['MethodLabel'] == 'Physician') & (csv_table['Projection'] == 'PA')]
print('Method Label="Physician" and Projection="PA":', len(csv_filtered))

Method Label="Physician" and Projection="PA": 23961


In [46]:
# Filter the dataset to only include rows where the 'Labels' column has a certain number of occurrences of the string 'normal'
normal_labels = [row for row in csv_filtered['Labels'] if row.count("'normal'") == len(eval(row))]
csv_filtered_normal = csv_filtered[csv_filtered['Labels'].isin(normal_labels)]
print('Method Label="Physician", Projection="PA" and Labels="normal":', len(csv_filtered_normal))

Method Label="Physician", Projection="PA" and Labels="normal": 8708


In [47]:
# Filter the dataset to only include rows where the 'Labels' column include a label different from 'normal'
abnormal_labels = [row for row in csv_filtered['Labels'] if  row.count("'normal'") != len(eval(row))]
csv_filtered_abnormal = csv_filtered[csv_filtered['Labels'].isin(abnormal_labels)]
print('Method Label="Physician", Projection="PA" and Labels="abnormal":', len(csv_filtered_abnormal))

Method Label="Physician", Projection="PA" and Labels="abnormal": 15253
